
In this notebook, we connect to the official ISTAT API to automatically fetch the latest available dataset on traffic accidents, defining a function for evenctual future API calls.

In [1]:
import requests
import pandas as pd

def fetch_istat_data(start_year):
#the url is generated from the official ISTAT API swagger
    url = f"https://esploradati.istat.it/SDMXWS/rest/data/41_983/all/?startPeriod={start_year}"
    header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}
    response = requests.get(url, headers=header)
        
    if response.status_code == 200:
        df1 = pd.read_csv(url, storage_options=header)
        return df1
    else:
        print(f"Error. Status code:{response.status_code}")

Now, we can fetch the actual data. We decided to focus on data starting from 2014, although the original dataset contains data points dating back to 2001 (checked with df_traffic_incidents["TIME_PERIOD"].min()). This decision is based on the assumption that technologies and regulations between 2001 and 2014 were different from current ones. By narrowing the scope to the last decade, we can establish a historical baseline while focusing on recent and relevant data. This timeframe also accounts fot the covid pandemic period, where we expect to observe some variations in traffic and incidents trends.

In [2]:
df_traffic_incidents = fetch_istat_data(2014)


In [3]:
#general overview
df_traffic_incidents

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2014,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2015,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2016,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2017,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2018,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257710,IT1:41_983(1.0),A,111107,ROADACC,9,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
257711,IT1:41_983(1.0),A,111107,ROADACC,9,2021,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
257712,IT1:41_983(1.0),A,111107,ROADACC,9,2022,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
257713,IT1:41_983(1.0),A,111107,ROADACC,9,2023,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_traffic_incidents.shape

(257715, 16)

In [5]:
df_traffic_incidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 257715 entries, 0 to 257714
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          257715 non-null  str    
 1   FREQ              257715 non-null  str    
 2   REF_AREA          257715 non-null  int64  
 3   DATA_TYPE         257715 non-null  str    
 4   RESULT            257715 non-null  str    
 5   TIME_PERIOD       257715 non-null  int64  
 6   OBS_VALUE         257715 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

From the ".info()" method, we can see that colums 7-15 are not filled with data, so we can drop them. Additionally, based on official ISTAT documentation we mapped the remaining features as follows: 
- dataflow: contains the source identifier fo all rows. Not having any analytical value we can drop this column 
- freq: the frequency of data points. Since all data are aggregated yearly, this column provides no analytical values and can be removed
- ref_area: indicates the geographic area (the specific municipality)
- data_type: indicates the metric type. KILLINJ refers to the number of deaths and injuries ("Killed & Injured") while ROADACC refers to the total number of "Road Accidents"
- result: details the specific outcome of the event. "M" stands for the number of fatalities/deaths, "F" for numbers of injured people, "9" stands for the total number of road accidents
- time_period: the year the data was recorded
- obs_value: the actual numerical value observed (the count of accidents or casualties)

In [6]:
df_traffic_incidents_copy = df_traffic_incidents.copy()
df_traffic_incidents_copy.drop(columns=["DATAFLOW", "FREQ", "OBS_STATUS", "NOTE_DS", "NOTE_REF_AREA", "NOTE_DATA_TYPE", "NOTE_RESULT", "NOTE_TIME_PERIOD", "BASE_PER", "UNIT_MEAS", "UNIT_MULT"], inplace=True)
df_traffic_incidents_copy

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,1001,KILLINJ,F,2014,11
1,1001,KILLINJ,F,2015,9
2,1001,KILLINJ,F,2016,8
3,1001,KILLINJ,F,2017,0
4,1001,KILLINJ,F,2018,1
...,...,...,...,...,...
257710,111107,ROADACC,9,2020,2
257711,111107,ROADACC,9,2021,5
257712,111107,ROADACC,9,2022,1
257713,111107,ROADACC,9,2023,4


The remaining columns have the correct datatype, so we can leave them as they are.
Next, we can proceed to collect additional data from "https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31" since the "ref_area" column needs to be decoded into actual municipality names. We will also search for supplemental information, such as population data and other relevant indicators for our analysis. To automate this data collection process, we can web scrape the website using the Selenium library.

In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=chrome_options)
driver.get("https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2024-12-31") #reference date set to 31-12-2024 to match the latest available ISTAT municipal data registry.

In [9]:
import time

driver.find_element(By.CLASS_NAME, "ant-select-selection-item").click()
time.sleep(3) #applied a delay to ensure the asynchronous dropdown menu fully renders
driver.find_element(By.XPATH, '//*[@id="tableid"]/div/div[2]/div/div/nav/ul/li[11]/div/div[2]/div/div/div/div[2]/div/div/div/div[4]').click()
time.sleep(3)

page = 1
df_temp = pd.DataFrame()

while page <= 79:
    rows = driver.find_elements(By.XPATH, "//table//tbody/tr" )
    page_list = []
    for r in rows:
        cells = r.find_elements(By.TAG_NAME, "td")
        cell_text = [c.text for c in cells]
        page_list.append(cell_text)
    
    df_page = pd.DataFrame(page_list)
    df_temp = pd.concat([df_temp, df_page], ignore_index=True)
    if page <79:
        driver.find_element(By.XPATH, "//button[@title='Pagina successiva']").click()
        time.sleep(10) #page transition delay increased to 10 seconds after empirical testing
        
    page = page + 1
    
    

In [10]:
df_municipalities = df_temp #renaming the generic temporary DataFrame to a descriptive name
df_municipalities

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,,,,,,,,,,,,,,,,,
1,1,01,001,201,001001,1001,Agliè,,TO,0,0,2562,2021,"13,1463",2024,2585,2024
2,1,01,001,201,001002,1002,Airasca,,TO,0,0,3660,2021,"15,7393",2024,3695,2024
3,1,01,001,201,001003,1003,Ala di Stura,,TO,0,0,467,2021,"46,3316",2024,463,2024
4,1,01,001,201,001004,1004,Albiano d'Ivrea,,TO,0,0,1637,2021,"11,7397",2024,1624,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7970,5,20,111,111,111103,111103,Villaputzu,,SU,0,0,4509,2021,"181,3947",2024,4368,2024
7971,5,20,111,111,111104,111104,Villasalto,,SU,0,0,989,2021,"130,3596",2024,903,2024
7972,5,20,111,111,111105,111105,Villasimius,,SU,0,0,3705,2021,"58,1781",2024,3735,2024
7973,5,20,111,111,111106,111106,Villasor,,SU,0,0,6599,2021,"86,8137",2024,6534,2024


In [11]:
df_municipalities.info()

<class 'pandas.DataFrame'>
RangeIndex: 7975 entries, 0 to 7974
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       7975 non-null   str  
 1   1       7975 non-null   str  
 2   2       7975 non-null   str  
 3   3       7975 non-null   str  
 4   4       7975 non-null   str  
 5   5       7975 non-null   str  
 6   6       7975 non-null   str  
 7   7       7975 non-null   str  
 8   8       7975 non-null   str  
 9   9       7975 non-null   str  
 10  10      7975 non-null   str  
 11  11      7975 non-null   str  
 12  12      7975 non-null   str  
 13  13      7975 non-null   str  
 14  14      7975 non-null   str  
 15  15      7975 non-null   str  
 16  16      7975 non-null   str  
dtypes: str(17)
memory usage: 1.0 MB


In [12]:
#several empty rows were detected due to the multi-page scraping process. We filter them out based on the logical premise that column 6 (Municipality Name) must contain a valid string with a length of at least 1 character. 
df_municipalities = df_municipalities[df_municipalities[6].str.len() > 0]

In [13]:
#dropping columns with redundant administrative codes, retaining only core geographical and structural variables, prioritizing the official Legal Population as our demographic baseline to ensure
#institutional data alignment.
df_municipalities_clean = df_municipalities[[1,5,6,11,13]]

In [14]:
#assigning official column names based on the source SITAS table
df_municipalities_clean.columns = ["Codice_Regione", "Codice_Comune", "Comune", "Popolazione_Legale", "Superficie(Kmq)"]

In [15]:
#resetting the index, validating and changing data types, performing a final structural check for any remaining NaN values.
df_municipalities_clean.reset_index(drop=True)
df_municipalities_clean["Superficie(Kmq)"] = df_municipalities_clean["Superficie(Kmq)"].astype(str).str.replace(',', '.')
df_municipalities_clean = df_municipalities_clean.astype({
    "Codice_Regione": "int64",
    "Codice_Comune": "int64",
    "Comune": "str",
    "Popolazione_Legale": "int64",
    "Superficie(Kmq)": "float64"
    })

In [16]:
df_municipalities_clean.info()

<class 'pandas.DataFrame'>
Index: 7896 entries, 1 to 7974
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Codice_Regione      7896 non-null   int64  
 1   Codice_Comune       7896 non-null   int64  
 2   Comune              7896 non-null   str    
 3   Popolazione_Legale  7896 non-null   int64  
 4   Superficie(Kmq)     7896 non-null   float64
dtypes: float64(1), int64(3), str(1)
memory usage: 370.1 KB


To enrich the "df_municipalities_clean" dataset with official regional names, we identify a secondary reference table within the SITAS portal. 
Given the strict execution timelines and the high latency observed during dynamic multi-page scraping, we adopt different strategy. Instead of programmatically iterating through the web interface, we'll use the portal's native "Export" functionality to retrieve the raw data. 

This file will be subsequently loaded into our environment, audited, and cleaned.

In [17]:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--disable-gpu")
driver2 = webdriver.Chrome(options=chrome_options)
driver2.get("https://situas.istat.it/web/#/territorio/body?id=61&dateFrom=2024-12-31")
time.sleep(3)
driver2.find_element(By.CLASS_NAME, "mr-2").click()
time.sleep(3)
driver2.find_element(By.XPATH, '//*[@id="dati-report-export-btn-menu"]/li[2]/button').click()


In [ ]:
df_temp2 = pd.read_csv("data/raw_data/raw_municipalities_data.csv", sep=";")

In [19]:
df_temp2

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione italiana),Comune (dizione straniera),Ripartizione geografica,...,Provincia/Uts,Tipologia Provincia/Uts,Capoluogo di regione,Capoluogo di provincia/uts,Sigla automobilistica,Codice catasto,Codice fiscale,Codice NUTS1 2024,Codice NUTS2 2024,Codice NUTS3 2024
0,1,1,1,201,1001,1001,Agliè,Agliè,NaN,Nord-ovest,...,Torino,3,0,0,TO,A074,83501790014,ITC,ITC1,ITC11
1,1,1,1,201,1002,1002,Airasca,Airasca,NaN,Nord-ovest,...,Torino,3,0,0,TO,A109,85002910017,ITC,ITC1,ITC11
2,1,1,1,201,1003,1003,Ala di Stura,Ala di Stura,NaN,Nord-ovest,...,Torino,3,0,0,TO,A117,83002970016,ITC,ITC1,ITC11
3,1,1,1,201,1004,1004,Albiano d'Ivrea,Albiano d'Ivrea,NaN,Nord-ovest,...,Torino,3,0,0,TO,A157,1735420018,ITC,ITC1,ITC11
4,1,1,1,201,1006,1006,Almese,Almese,NaN,Nord-ovest,...,Torino,3,0,0,TO,A218,1817670019,ITC,ITC1,ITC11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7891,5,20,111,111,111103,111103,Villaputzu,Villaputzu,NaN,Isole,...,Sud Sardegna,1,0,0,SU,L998,80003170927,ITG,ITG2,ITG2H
7892,5,20,111,111,111104,111104,Villasalto,Villasalto,NaN,Isole,...,Sud Sardegna,1,0,0,SU,M016,1391410923,ITG,ITG2,ITG2H
7893,5,20,111,111,111105,111105,Villasimius,Villasimius,NaN,Isole,...,Sud Sardegna,1,0,0,SU,B738,80014170924,ITG,ITG2,ITG2H
7894,5,20,111,111,111106,111106,Villasor,Villasor,NaN,Isole,...,Sud Sardegna,1,0,0,SU,M025,82002160925,ITG,ITG2,ITG2H


In [20]:
#Extracting the Region name and joining it with the primary clean dataset using the unique municipality code as the relational key.
df_temp2 = df_temp2[["Codice Comune (numerico)", "Regione"]]
df_temp2 = df_temp2.rename(columns={"Codice Comune (numerico)":"Codice_Comune"})
df_municipalities_clean = df_municipalities_clean.merge(df_temp2, on="Codice_Comune", how="left")

In [21]:
df_municipalities_clean

,Codice_Regione,Codice_Comune,Comune,Popolazione_Legale,Superficie(Kmq),Regione
0,1,1001,Agliè,2562,13.1463,Piemonte
1,1,1002,Airasca,3660,15.7393,Piemonte
2,1,1003,Ala di Stura,467,46.3316,Piemonte
3,1,1004,Albiano d'Ivrea,1637,11.7397,Piemonte
4,1,1006,Almese,6331,17.8741,Piemonte
...,...,...,...,...,...,...
7891,20,111103,Villaputzu,4509,181.3947,Sardegna
7892,20,111104,Villasalto,989,130.3596,Sardegna
7893,20,111105,Villasimius,3705,58.1781,Sardegna
7894,20,111106,Villasor,6599,86.8137,Sardegna


We now can integrate official data regarding the registered vehicle fleet ("Parco macchine") updated to 2024, sourced from the Italian Automobile Club (ACI). 
While highly granular municipality-level fleet data exists within the ACI registries, it is subject to commercial licensing fees and paywalls. Consequently, we intentionally leverage the officially available Open Data at the regional level for establishing a comparative baseline to capture and evaluate the overall volume and density of vehicles distributed across each Italian region.

In [ ]:
df_aci = pd.read_csv("data/raw_data/aci_open_data_fleet_2024.csv", sep=";")
df_aci

,CONSISTENZA DEL PARCO AUTOVETTURE NELLE REGIONI,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tab.III.9
1,REGIONI,2000,2005,2010,2015,2020,2022,2024,2000,2005,2010,2015,2020,2022,2024
2,NaN,Valori assoluti,NaN,NaN,NaN,NaN,NaN,NaN,Numeri indice,NaN,NaN,NaN,NaN,NaN,NaN
3,Piemonte,2.635.135,2.703.252,2.782.541,2.844.680,2.915.687,2.900.449,3.075.162,"100,0","102,6","105,6","108,0","110,6","110,1","116,7"
4,Valle D'Aosta,128.007,131.960,134.836,145.266,221.721,287.951,237.162,"100,0","103,1","105,3","113,5","173,2","224,9","185,3"
5,Lombardia,5.285.721,5.555.076,5.808.621,5.923.849,6.231.939,6.272.187,6.426.326,"100,0","105,1","109,9","112,1","117,9","118,7","121,6"
6,Trentino A.A.,499.130,530.570,566.833,885.769,1.162.970,1.276.378,1.291.248,"100,0","106,3","113,6","177,5","233,0","255,7","258,7"
7,Veneto,2.607.903,2.782.469,2.939.099,3.011.316,3.198.100,3.221.693,3.302.750,"100,0","106,7","112,7","115,5","122,6","123,5","126,6"
8,Friuli V.G.,700.705,734.233,763.144,773.619,808.422,812.503,828.929,"100,0","104,8","108,9","110,4","115,4","116,0","118,3"
9,Liguria,821.275,823.377,841.795,828.022,845.474,843.142,850.098,"100,0","100,3","102,5","100,8","102,9","102,7","103,5"


In [ ]:
#Extracting and cleaning the 2024 regional car fleet data from the raw ACI dataset.
#Filtering for relevant columns, isolating regional rows (excluding headers/totals)
df_aci = df_aci[["CONSISTENZA DEL PARCO AUTOVETTURE NELLE REGIONI", "Unnamed: 7"]]
df_aci = df_aci.iloc[3:23]
df_aci.columns = ["Regione", "Parco_Auto_2024"]
df_aci = df_aci.reset_index(drop=True)

In [49]:
df_aci

,Regione,Parco_Auto_2024
0,Piemonte,3.075.162
1,Valle D'Aosta,237.162
2,Lombardia,6.426.326
3,Trentino A.A.,1.291.248
4,Veneto,3.302.750
5,Friuli V.G.,828.929
6,Liguria,850.098
7,Emilia Romagna,3.082.830
8,Toscana,2.722.906
9,Umbria,659.203


We can now merge the municipality names to the core Istat dataset. Then, we can proceed to exporting the datasets for future use (EDA, dashboard creation with Power BI, etc.)

In [50]:
df_traffic_incidents_copy = df_traffic_incidents_copy.merge(df_municipalities_clean, left_on="REF_AREA", right_on="Codice_Comune", how="left")
df_traffic_incidents_copy

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Codice_Regione,Codice_Comune,Comune,Popolazione_Legale,Superficie(Kmq),Regione
0,1001,KILLINJ,F,2014,11,1.0,1001.0,Agliè,2562.0,13.1463,Piemonte
1,1001,KILLINJ,F,2015,9,1.0,1001.0,Agliè,2562.0,13.1463,Piemonte
2,1001,KILLINJ,F,2016,8,1.0,1001.0,Agliè,2562.0,13.1463,Piemonte
3,1001,KILLINJ,F,2017,0,1.0,1001.0,Agliè,2562.0,13.1463,Piemonte
4,1001,KILLINJ,F,2018,1,1.0,1001.0,Agliè,2562.0,13.1463,Piemonte
...,...,...,...,...,...,...,...,...,...,...,...
257710,111107,ROADACC,9,2020,2,20.0,111107.0,Villaspeciosa,2536.0,27.1943,Sardegna
257711,111107,ROADACC,9,2021,5,20.0,111107.0,Villaspeciosa,2536.0,27.1943,Sardegna
257712,111107,ROADACC,9,2022,1,20.0,111107.0,Villaspeciosa,2536.0,27.1943,Sardegna
257713,111107,ROADACC,9,2023,4,20.0,111107.0,Villaspeciosa,2536.0,27.1943,Sardegna


In [52]:
df_traffic_incidents_copy = df_traffic_incidents_copy.drop(columns=["Codice_Comune", "Popolazione_Legale", "Superficie(Kmq)"])


In [53]:
df_traffic_incidents_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 257715 entries, 0 to 257714
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   REF_AREA        257715 non-null  int64  
 1   DATA_TYPE       257715 non-null  str    
 2   RESULT          257715 non-null  str    
 3   TIME_PERIOD     257715 non-null  int64  
 4   OBS_VALUE       257715 non-null  int64  
 5   Codice_Regione  253314 non-null  float64
 6   Comune          253314 non-null  str    
 7   Regione         253314 non-null  str    
dtypes: float64(1), int64(3), str(4)
memory usage: 15.7 MB


In [55]:
municipalities_null = df_traffic_incidents_copy[df_traffic_incidents_copy["Comune"].isna()]["REF_AREA"].unique()
municipalities_null

array([  1005,   1138,   1151,   1182,   1277,   1297,   2019,   2038,
         2111,   2112,   2114,   2123,   3071,   3157,   4036,   4048,
         4236,   5079,   5110,   6006,   6042,   6064,   6080,   6089,
         6130,   8013,   8036,  12009,  12018,  12028,  12095,  12111,
        13038,  13050,  13060,  13061,  13093,  13109,  13122,  13175,
        13179,  13194,  13199,  13205,  13215,  13228,  14042,  15235,
        15246,  17154,  18002,  18028,  18056,  18070,  18132,  18170,
        19008,  19042,  19071,  20004,  20006,  20009,  20023,  20040,
        20049,  20067,  22004,  22012,  22019,  22020,  22023,  22024,
        22027,  22028,  22030,  22031,  22041,  22046,  22055,  22056,
        22057,  22063,  22066,  22067,  22069,  22070,  22072,  22073,
        22075,  22076,  22077,  22080,  22082,  22084,  22086,  22088,
        22094,  22096,  22099,  22100,  22101,  22105,  22111,  22121,
        22122,  22125,  22126,  22132,  22140,  22145,  22146,  22148,
      

In [59]:
#Some records correspond to ceased or merged municipalities. To preserve these data points for analysis, missing names are filled with "Comune cessato/fuso"
df_traffic_incidents_copy["Comune"] = df_traffic_incidents_copy["Comune"].fillna("Comune cessato/fuso")
df_traffic_incidents_copy["Regione"] = df_traffic_incidents_copy["Regione"].fillna("Comune cessato/fuso")
df_traffic_incidents_copy["Codice_Regione"] = df_traffic_incidents_copy["Codice_Regione"].fillna(0).astype(int)

In [62]:
df_traffic_incidents_copy.to_csv("data/traffic_incidents.csv", index=False)
df_municipalities_clean.to_csv("data/municipalities_metrics.csv", index=False)
df_aci.to_csv("data/aci_fleet.csv", index=False)